In [ ]:
kafka_topic = "store_operation"
retention_hours = 72
starting_offsets = "earliest"  # first run; the checkpoint takes over afterwards


In [ ]:
# rlt_ingest_sale_product — REALTIME speed layer for eod_sale_product.
# Micro-batch (Trigger.AvailableNow): read new Kafka offsets, upsert, stop. Schedule it
# every few minutes → near-realtime at batch cost. Fabric equivalent of ClickHouse
# rlt_sale_product_mv + eod_sale_product_view_rlt.
#
# PREREQUISITES (set as Environment Spark conf secrets, resolver maps '-'->'_'):
#   kafka-bootstrap  Aiven "Apache Kafka" host:port (NOT the REST 14404 port)
#   kafka-user / kafka-pass   SASL SCRAM creds (enable SASL on the Aiven service)
#   kafka-ca         Aiven CA cert (PEM text)
# Also: the Spark-Kafka connector (spark-sql-kafka-0-10) must be available in the runtime,
# and gold.fact_eod_sale_product (the batch fact) must exist for the UNION view.
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, ArrayType
from delta.tables import DeltaTable

def get_secret(name):
    return spark.conf.get("spark.ssv.secret." + name.replace("-", "_"))

_ca_path = "/tmp/aiven_ca.pem"
with open(_ca_path, "w") as _f:
    _f.write(get_secret("kafka-ca"))

KAFKA = {
    "kafka.bootstrap.servers": get_secret("kafka-bootstrap"),
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "SCRAM-SHA-256",
    "kafka.sasl.jaas.config": ('org.apache.kafka.common.security.scram.ScramLoginModule required '
                               f'username="{get_secret("kafka-user")}" password="{get_secret("kafka-pass")}";'),
    "kafka.ssl.truststore.type": "PEM",
    "kafka.ssl.truststore.location": _ca_path,
}


In [ ]:
VN_OFFSET_H = 7  # document_date is epoch ms (UTC); VN business day = +7h (matches batch)

ITEM = StructType([
    StructField("product_code", StringType()), StructField("product_name", StringType()),
    StructField("uom", StringType()), StructField("product_uom_id", LongType()),
    StructField("uom_size", LongType()), StructField("quantity", LongType()),
    StructField("retail_selling_price", DoubleType()),
    StructField("total_amount", DoubleType()), StructField("total_amount_without_tax", DoubleType()),
    StructField("purchase_price_with_tax", DoubleType()), StructField("purchase_price_without_tax", DoubleType()),
    StructField("product_group_id", LongType()), StructField("product_group", StringType()),
    StructField("product_sub_category_id", LongType()), StructField("product_sub_category", StringType()),
    StructField("product_category_id", LongType()), StructField("product_category", StringType()),
    StructField("retail_business_type", StringType()),
    StructField("supplier_code", StringType()), StructField("supplier_name", StringType())])
TXN = StructType([
    StructField("transaction_id", StringType()), StructField("staff_id", StringType()),
    StructField("staff_name", StringType()), StructField("document_date", LongType()),
    StructField("posting_date", LongType()), StructField("payment_method", LongType()),
    StructField("total_amount", DoubleType()), StructField("customer_gender", LongType()),
    StructField("sale_normal_items", ArrayType(ITEM))])
ENVELOPE = StructType([
    StructField("stream_type", StringType()), StructField("source", StringType()),
    StructField("payload", StructType([
        StructField("store_code", StringType()), StructField("store_name", StringType()),
        StructField("sale_transaction", ArrayType(TXN))]))])

# rlt cols = subset of fact_eod_sale_product so the batch+rlt UNION view aligns
RLT_COLS = ["transaction_id","product_id","report_date","transaction_time","sale_bill_time",
            "posting_time","store_id","store_name","staff_id","staff_name","payment_method_id",
            "product_name","product_uom","product_uom_id","product_group_id","product_group_name",
            "product_sub_category_id","product_sub_category_name","product_category_id",
            "product_category_name","retail_business_type","supplier_id","supplier_name",
            "product_uom_size","quantity","product_price","total_amount","total_amount_no_tax",
            "final_amount","final_amount_no_tax","purchase_price_with_tax","purchase_price_without_tax",
            "customer_gender","transaction_type","delivery_status"]

def conform_store_operation(df):
    """Kafka value (JSON) -> rlt line grain (txn x item). Mirrors the ClickHouse rlt MV."""
    df = df.withColumn("e", F.from_json(F.col("value"), ENVELOPE)).where(F.col("e.stream_type") == "SALE_TRANSACTION")
    df = df.withColumn("t", F.explode("e.payload.sale_transaction")).withColumn("it", F.explode("t.sale_normal_items"))
    ttime = F.to_timestamp(F.col("t.document_date")/1000) + F.expr(f"INTERVAL {VN_OFFSET_H} HOURS")
    ptime = F.to_timestamp(F.col("t.posting_date")/1000) + F.expr(f"INTERVAL {VN_OFFSET_H} HOURS")
    g = F.col("t.customer_gender")
    return df.select(
        F.col("t.transaction_id").alias("transaction_id"), F.col("it.product_code").alias("product_id"),
        F.to_date(ttime).alias("report_date"), ttime.alias("transaction_time"),
        ttime.alias("sale_bill_time"), ptime.alias("posting_time"),
        F.col("e.payload.store_code").alias("store_id"), F.col("e.payload.store_name").alias("store_name"),
        F.col("t.staff_id").alias("staff_id"), F.col("t.staff_name").alias("staff_name"),
        F.col("t.payment_method").cast("int").alias("payment_method_id"),
        F.col("it.product_name").alias("product_name"), F.col("it.uom").alias("product_uom"),
        F.col("it.product_uom_id").cast("int").alias("product_uom_id"),
        F.col("it.product_group_id").cast("int").alias("product_group_id"),
        F.col("it.product_group").alias("product_group_name"),
        F.col("it.product_sub_category_id").cast("int").alias("product_sub_category_id"),
        F.col("it.product_sub_category").alias("product_sub_category_name"),
        F.col("it.product_category_id").cast("int").alias("product_category_id"),
        F.col("it.product_category").alias("product_category_name"),
        F.col("it.retail_business_type").alias("retail_business_type"),
        F.col("it.supplier_code").alias("supplier_id"), F.col("it.supplier_name").alias("supplier_name"),
        F.col("it.uom_size").cast("int").alias("product_uom_size"),
        F.col("it.quantity").cast("int").alias("quantity"),
        F.col("it.retail_selling_price").cast("float").alias("product_price"),
        F.col("it.total_amount").cast("float").alias("total_amount"),
        F.col("it.total_amount_without_tax").cast("float").alias("total_amount_no_tax"),
        F.col("it.total_amount").cast("float").alias("final_amount"),
        F.col("it.total_amount_without_tax").cast("float").alias("final_amount_no_tax"),
        F.col("it.purchase_price_with_tax").cast("float").alias("purchase_price_with_tax"),
        F.col("it.purchase_price_without_tax").cast("float").alias("purchase_price_without_tax"),
        F.when(g == 1, "Male").when(g == 2, "Female").otherwise("").alias("customer_gender"),
        F.lit("Sale Transaction").alias("transaction_type"), F.lit("completed").alias("delivery_status"))


In [ ]:
TARGET = "gold.rlt_fact_eod_sale_product"
KEYS = ["transaction_id", "product_id", "product_uom_id"]
CKPT = "Files/checkpoints/rlt_sale_product"

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
if not spark.catalog.tableExists(TARGET):
    (conform_store_operation(spark.createDataFrame([], "value string"))
        .withColumn("inserted_at", F.current_timestamp())
        .write.format("delta").mode("overwrite").saveAsTable(TARGET))
    print("created empty", TARGET)

def upsert(batch_df, batch_id):
    rlt = conform_store_operation(batch_df).withColumn("inserted_at", F.current_timestamp())
    w = Window.partitionBy(*KEYS).orderBy(F.col("inserted_at").desc())     # latest per key in batch
    rlt = rlt.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")
    cond = " AND ".join(f"t.{k} = s.{k}" for k in KEYS)
    (DeltaTable.forName(spark, TARGET).alias("t")
        .merge(rlt.alias("s"), cond).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

(spark.readStream.format("kafka").options(**KAFKA)
    .option("subscribe", kafka_topic).option("startingOffsets", starting_offsets)
    .option("failOnDataLoss", "false").load()
    .selectExpr("CAST(value AS STRING) AS value")
    .writeStream.foreachBatch(upsert).option("checkpointLocation", CKPT)
    .trigger(availableNow=True).start().awaitTermination())

# speed layer keeps only the recent window
spark.sql(f"DELETE FROM {TARGET} WHERE transaction_time < current_timestamp() - INTERVAL {retention_hours} HOURS")
print("rlt rows after upsert+retention:", spark.table(TARGET).count())


In [ ]:
# Serving lambda view: batch (past days) + realtime (today, not yet batched)
cols = ", ".join(RLT_COLS)
spark.sql(f"""
CREATE OR REPLACE VIEW gold.vw_eod_sale_product_rlt AS
SELECT {cols}, 'eod' AS source FROM gold.fact_eod_sale_product
  WHERE report_date <= (SELECT max(report_date) FROM gold.fact_eod_sale_product)
UNION ALL
SELECT {cols}, 'rlt' AS source FROM {TARGET}
  WHERE report_date > (SELECT coalesce(max(report_date), DATE'1900-01-01') FROM gold.fact_eod_sale_product)
""")
rlt_today = spark.sql("SELECT count(*) c FROM gold.vw_eod_sale_product_rlt WHERE source='rlt'").first().c
print("view gold.vw_eod_sale_product_rlt ready — realtime slice rows:", rlt_today)
